In [ ]:
!pip install -q vllm
!pip uninstall -y torchaudio torchcodec

# Restart kernel otomatis supaya perubahan library terbaca
import os
os.kill(os.getpid(), 9)

Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0
Found existing installation: torchcodec 0.17.0
Uninstalling torchcodec-0.17.0:
  Successfully uninstalled torchcodec-0.17.0


In [1]:
!python -c "import torch, transformers, vllm; print('torch', torch.__version__, '| transformers', transformers.__version__, '| vllm', vllm.__version__, '| GPU', torch.cuda.device_count())"

torch 2.13.0+cu130 | transformers 5.16.1 | vllm 0.31.0 | GPU 2


In [2]:
%%writefile /kaggle/working/01_label_reports.py
import os, json, re
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

TRAIN_CSV = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"
OUT_CSV   = "/kaggle/working/report_labels.csv"
MODEL_ID  = "Qwen/Qwen2.5-7B-Instruct"
# Kalau kena OOM, ganti ke versi 4-bit: "Qwen/Qwen2.5-7B-Instruct-AWQ"
MAX_CHARS = 4000
CHUNK     = 500

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]

DEFS = """
ACL: anterior cruciate ligament tear (partial or complete) or clear ACL injury.
MCL: medial collateral ligament injury (sprain or tear).
Medial Meniscus: tear of the medial meniscus (not mere degeneration without tear).
Lateral Meniscus: tear of the lateral meniscus (not mere degeneration without tear).
Medial OA: osteoarthritis / significant chondral loss in the medial femorotibial compartment.
Lateral OA: osteoarthritis / significant chondral loss in the lateral femorotibial compartment.
PF OA: osteoarthritis / chondropathy of the patellofemoral compartment (patella or trochlea).
Effusion: joint effusion that is reported as present (not 'no effusion').
Synovitis: synovitis or synovial thickening/hypertrophy.
Baker's: Baker's cyst / popliteal cyst.
Contusion: bone contusion / bone bruise / bone marrow edema from trauma.
Fracture: any fracture (including avulsion or osteochondral fracture).
"""

SYSTEM = (
    "You are an expert musculoskeletal radiologist. You read knee MRI reports "
    "written in any language and extract findings. Answer ONLY with JSON."
)

def build_prompt(report: str) -> str:
    keys = ", ".join(f'"{l}"' for l in LABELS)
    return (
        f"Label definitions:\n{DEFS}\n"
        "For each label, answer one of: \"positive\", \"uncertain\", \"negative\".\n"
        "Use \"uncertain\" for hedged findings (suspected, possible, R/O, cannot exclude).\n"
        "If a finding is not mentioned, answer \"negative\".\n"
        f"Return a JSON object with exactly these keys: {keys}.\n\n"
        f"REPORT:\n{report[:MAX_CHARS]}"
    )

MAP = {"positive": 1.0, "uncertain": 0.5, "negative": 0.0}

def parse(text: str) -> dict:
    m = re.search(r"\{.*\}", text, re.S)
    out = {l: np.nan for l in LABELS}
    if not m:
        return out
    try:
        d = json.loads(m.group(0))
    except json.JSONDecodeError:
        return out
    for l in LABELS:
        v = str(d.get(l, "")).strip().lower()
        out[l] = MAP.get(v, np.nan)
    return out

def evaluate(df, pred):
    gold = df[df[LABELS].notna().all(axis=1)][["StudyInstanceUID"] + LABELS]
    m = gold.merge(pred, on="StudyInstanceUID", suffixes=("_gold", "_llm"))
    print(f"\nEvaluasi pada {len(m)} studi berlabel emas")
    aucs = []
    for l in LABELS:
        y, p = m[f"{l}_gold"], m[f"{l}_llm"].fillna(0.5)
        auc = roc_auc_score(y, p) if y.nunique() > 1 else np.nan
        acc = ((p >= 0.5).astype(int) == y).mean()
        aucs.append(auc)
        print(f"{l:18s} AUC={auc:.3f}  Acc={acc:.3f}  pos_gold={int(y.sum())}")
    print(f"\nMean AUC (laporan vs emas): {np.nanmean(aucs):.3f}")

def main():
    df = pd.read_csv(TRAIN_CSV)

    done = pd.read_csv(OUT_CSV) if os.path.exists(OUT_CSV) else pd.DataFrame()
    done_ids = set(done["StudyInstanceUID"]) if len(done) else set()
    todo = df[~df["StudyInstanceUID"].isin(done_ids)].reset_index(drop=True)
    print(f"Sudah selesai: {len(done_ids)}, sisa: {len(todo)}")

    if len(todo):
        tok = AutoTokenizer.from_pretrained(MODEL_ID)
        llm = LLM(
            model=MODEL_ID,
            dtype="float16",
            tensor_parallel_size=2,
            max_model_len=4096,
            gpu_memory_utilization=0.90,
        )
        sp = SamplingParams(temperature=0.0, max_tokens=200)

        rows = done.to_dict("records") if len(done) else []
        for s in range(0, len(todo), CHUNK):
            chunk = todo.iloc[s:s + CHUNK]
            prompts = [
                tok.apply_chat_template(
                    [{"role": "system", "content": SYSTEM},
                     {"role": "user", "content": build_prompt(r)}],
                    tokenize=False, add_generation_prompt=True)
                for r in chunk["Report"].fillna("")
            ]
            outs = llm.generate(prompts, sp)
            for uid, o in zip(chunk["StudyInstanceUID"], outs):
                rows.append({"StudyInstanceUID": uid, **parse(o.outputs[0].text)})
            pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
            print(f"Tersimpan {len(rows)}/{len(df)}", flush=True)

    pred = pd.read_csv(OUT_CSV)
    print("Gagal parse:", pred[LABELS].isna().any(axis=1).sum(), "studi")
    evaluate(df, pred)

if __name__ == "__main__":
    main()

Overwriting /kaggle/working/01_label_reports.py


In [3]:
!python /kaggle/working/01_label_reports.py

Sudah selesai: 0, sisa: 4407
config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 2.24MB/s]
tokenizer_config.json: 100%|███████████████| 7.30k/7.30k [00:00<00:00, 15.8MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 47.5MB/s]
merges.txt: 100%|███████████████████████████| 1.67M/1.67M [00:00<00:00, 124MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 144MB/s]
(MainProcess pid=399) INFO 10-06 12:49:03 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
(MainProcess pid=399) INFO 10-06 12:49:18 [model.py:699] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=399) WARNING 10-06 12:49:18 [model.py:2442] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=399) INFO 10-06 12:49:18 [model.py:2109] Using max model len 4096
(MainProcess pid=399) INFO 10-06 12:49:

In [4]:
%%writefile /kaggle/working/02_retry_failed.py
import importlib.util, json, re
import numpy as np, pandas as pd
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

spec = importlib.util.spec_from_file_location("lab", "/kaggle/working/01_label_reports.py")
lab = importlib.util.module_from_spec(spec); spec.loader.exec_module(lab)
L = lab.LABELS

def parse_lenient(text):
    out = {l: np.nan for l in L}
    m = re.search(r"\{.*\}", text.replace("\\'", "'"), re.S)
    if not m:
        return out
    try:
        d = json.loads(m.group(0))
    except json.JSONDecodeError:
        return out
    norm = {re.sub(r"[^a-z]", "", k.lower()): v for k, v in d.items()}
    for l in L:
        v = str(norm.get(re.sub(r"[^a-z]", "", l.lower()), "")).strip().lower().rstrip(".")
        if v in ("positive", "yes", "present", "true", "1"):   out[l] = 1.0
        elif v in ("uncertain", "possible", "suspected", "0.5"): out[l] = 0.5
        elif v in ("negative", "no", "absent", "false", "0"):   out[l] = 0.0
    return out

df = pd.read_csv(lab.TRAIN_CSV)
pred = pd.read_csv(lab.OUT_CSV)
failed = pred.loc[pred[L].isna().any(axis=1), "StudyInstanceUID"]
todo = df[df["StudyInstanceUID"].isin(failed)].reset_index(drop=True)
print("Diproses ulang:", len(todo))

tok = AutoTokenizer.from_pretrained(lab.MODEL_ID)
llm = LLM(model=lab.MODEL_ID, dtype="float16", tensor_parallel_size=2,
          max_model_len=4096, gpu_memory_utilization=0.90)
prompts = [tok.apply_chat_template(
              [{"role": "system", "content": lab.SYSTEM},
               {"role": "user", "content": lab.build_prompt(r)}],
              tokenize=False, add_generation_prompt=True)
           for r in todo["Report"].fillna("")]
outs = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=400))

raw = [o.outputs[0].text for o in outs]
pd.DataFrame({"StudyInstanceUID": todo["StudyInstanceUID"], "raw": raw}).to_csv(
    "/kaggle/working/failed_raw.csv", index=False)

pred = pred.set_index("StudyInstanceUID")
for uid, t in zip(todo["StudyInstanceUID"], raw):
    pred.loc[uid, L] = pd.Series(parse_lenient(t))
pred = pred.reset_index()
pred.to_csv(lab.OUT_CSV, index=False)
print("Masih gagal:", pred[L].isna().any(axis=1).sum())
lab.evaluate(df, pred)

Writing /kaggle/working/02_retry_failed.py


In [5]:
!python /kaggle/working/02_retry_failed.py

Diproses ulang: 159
(MainProcess pid=972) INFO 10-06 13:18:03 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}
(MainProcess pid=972) INFO 10-06 13:18:03 [model.py:699] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=972) WARNING 10-06 13:18:03 [model.py:2442] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=972) INFO 10-06 13:18:03 [model.py:2109] Using max model len 4096
(MainProcess pid=972) INFO 10-06 13:18:03 [scheduler.py:308] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=972) INFO 10-06 13:18:03 [kernel.py:408] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
(EngineCore pid=999) INFO 10-06 13:18:07 [core.py:129] Initializing a V1 LLM engine (v0.31.0) with config:

In [9]:
import pandas as pd
L = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
     "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
     "Contusion", "Fracture"]
pred = pd.read_csv("/kaggle/working/report_labels.csv")
raw = pd.read_csv("/kaggle/working/failed_raw.csv")

print("Jumlah label kosong per kelas:")
print(pred[L].isna().sum().to_string())
print("\nStudi yang semua labelnya kosong:", pred[L].isna().all(axis=1).sum())

for t in raw["raw"].head(5):
    print("-----"); print(repr(str(t)[:700]))

Jumlah label kosong per kelas:
ACL                   4
MCL                  30
Medial Meniscus      28
Lateral Meniscus     28
Medial OA             8
Lateral OA            9
PF OA                14
Effusion             41
Synovitis           115
Baker's               3
Contusion             5
Fracture              3

Studi yang semua labelnya kosong: 0
-----
'{\n"ACL": "positive",\n"MCL": "distended",\n"Medial Meniscus": "partial tear",\n"Lateral Meniscus": "negative",\n"Medial OA": "negative",\n"Lateral OA": "suspected",\n"PF OA": "negative",\n"Effusion": "mild",\n"Synovitis": "mild",\n"Baker\'s": "negative",\n"Contusion": "negative",\n"Fracture": "osteochondral lesion"\n}'
-----
'{\n"ACL": "negative",\n"MCL": "intact",\n"Medial Meniscus": "tear",\n"Lateral Meniscus": "tear",\n"Medial OA": "positive",\n"Lateral OA": "negative",\n"PF OA": "positive",\n"Effusion": "positive",\n"Synovitis": "positive",\n"Baker\'s": "negative",\n"Contusion": "negative",\n"Fracture": "negative"\n}'
-----


In [11]:
import json, re, shutil
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

TRAIN = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"
OUT   = "/kaggle/working/report_labels.csv"
L = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
     "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
     "Contusion", "Fracture"]

shutil.copy(OUT, "/kaggle/working/report_labels_backup.csv")  # cadangan

UNCERTAIN = ["uncertain", "suspect", "possibl", "probabl", "r/o", "cannot", "equivocal", "likely"]
NEGATIVE  = ["negative", "intact", "normal", "none", "absent", "not ", "no ", "without", "unremarkable"]
POSITIVE  = ["positive", "present", "tear", "rupture", "mild", "moderate", "severe", "large",
             "small", "sprain", "grade", "edema", "oedema", "fract", "avulsion", "cyst",
             "chondro", "effusion", "synovitis", "lesion", "distended", "thicken"]

def map_val(v, label):
    if v is None:                      # key tidak ditulis -> dianggap tidak disebut
        return 0.0
    s = " " + str(v).lower().strip() + " "
    if s.strip() in ("no", "n/a", "na", "-"):
        return 0.0
    if any(k in s for k in UNCERTAIN):
        return 0.5
    if any(k in s for k in NEGATIVE):
        return 0.0
    if "Meniscus" in label and "degenerat" in s and "tear" not in s:
        return 0.0                     # sesuai definisi: degenerasi tanpa robekan = negatif
    if label == "Fracture" and not any(k in s for k in ["fract", "avulsion", "positive"]):
        return 0.5                     # misal "osteochondral lesion": belum tentu fraktur
    if any(k in s for k in POSITIVE):
        return 1.0
    return np.nan                      # benar-benar tidak dikenali -> tetap kosong

pred = pd.read_csv(OUT).set_index("StudyInstanceUID")
raw  = pd.read_csv("/kaggle/working/failed_raw.csv")
key  = lambda x: re.sub(r"[^a-z]", "", x.lower())

for uid, t in zip(raw["StudyInstanceUID"], raw["raw"]):
    m = re.search(r"\{.*\}", str(t).replace("\\'", "'"), re.S)
    if not m:
        continue
    try:
        d = json.loads(m.group(0))
    except json.JSONDecodeError:
        continue
    norm = {key(k): v for k, v in d.items()}
    for l in L:
        if pd.isna(pred.loc[uid, l]):
            pred.loc[uid, l] = map_val(norm.get(key(l)), l)

pred = pred.reset_index()
pred.to_csv(OUT, index=False)
print("Label masih kosong:", int(pred[L].isna().sum().sum()), "sel")
print("Studi dengan label kosong:", int(pred[L].isna().any(axis=1).sum()))

# Evaluasi ulang
df = pd.read_csv(TRAIN)
g = df[df["ACL"].notna()][["StudyInstanceUID"] + L].merge(
        pred, on="StudyInstanceUID", suffixes=("_gold", "_llm"))
aucs = []
for l in L:
    auc = roc_auc_score(g[f"{l}_gold"], g[f"{l}_llm"].fillna(0.5))
    aucs.append(auc)
    print(f"{l:18s} AUC={auc:.3f}")
print(f"\nMean AUC: {np.mean(aucs):.3f}")

Label masih kosong: 31 sel
Studi dengan label kosong: 18
ACL                AUC=0.890
MCL                AUC=0.939
Medial Meniscus    AUC=0.867
Lateral Meniscus   AUC=0.771
Medial OA          AUC=0.863
Lateral OA         AUC=0.810
PF OA              AUC=0.782
Effusion           AUC=0.677
Synovitis          AUC=0.674
Baker's            AUC=0.893
Contusion          AUC=0.753
Fracture           AUC=0.792

Mean AUC: 0.809


In [12]:
import os, pandas as pd
base = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

for f in ["train_series.csv", "test_series.csv", "test.csv", "sample_submission.csv"]:
    d = pd.read_csv(f"{base}/{f}")
    print(f"\n=== {f}  shape={d.shape}")
    print("Kolom:", d.columns.tolist())
    print(d.head(5).to_string(max_colwidth=60))

# Contoh struktur folder satu studi di train_series
root = f"{base}/train_series"
first = sorted(os.listdir(root))[0]
print(f"\n=== Struktur folder: train_series/{first}")
for dirpath, dirs, files in os.walk(os.path.join(root, first)):
    depth = dirpath.replace(root, "").count(os.sep)
    print("  " * depth + os.path.basename(dirpath) + "/", f"({len(files)} file)")
    for fn in sorted(files)[:3]:
        print("  " * (depth + 1) + fn)


=== train_series.csv  shape=(24371, 5)
Kolom: ['StudyInstanceUID', 'SeriesInstanceUID', 'Fluid_Sensitive', 'Fat_Suppression', 'Anatomical_Plane']
                                              StudyInstanceUID                                            SeriesInstanceUID  Fluid_Sensitive  Fat_Suppression Anatomical_Plane
0  1.2.826.0.1.3680043.8.498.100048732290990538690933242921...  1.2.826.0.1.3680043.8.498.123431101950362134834540917154...                1                1         Sagittal
1  1.2.826.0.1.3680043.8.498.100048732290990538690933242921...  1.2.826.0.1.3680043.8.498.138212297449972206415752919274...                1                1            Axial
2  1.2.826.0.1.3680043.8.498.100048732290990538690933242921...  1.2.826.0.1.3680043.8.498.230848365367225952758286902931...                0                0          Coronal
3  1.2.826.0.1.3680043.8.498.100048732290990538690933242921...  1.2.826.0.1.3680043.8.498.407342061024587230961546871473...                1             